In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS catalog.silver;

In [0]:
%sql

SELECT * FROM catalog.bronze.diagnosis_raw;

diagnosis_code,diagnosis_desc,diagnosis_category,readmission_risk_level,_rescued_data,_source_file,_ingest_timestamp,ingestion_date
I10,Essential Hypertension,Cardiovascular,Medium,null,abfss://data@datastoragea.dfs.core.windows.net/staging/diagnosis_raw/diagnosis_raw.csv,2026-05-04T13:33:37.059Z,2026-05-04
I50,Heart Failure,Cardiovascular,High,null,abfss://data@datastoragea.dfs.core.windows.net/staging/diagnosis_raw/diagnosis_raw.csv,2026-05-04T13:33:37.059Z,2026-05-04
I21,Acute Myocardial Infarction,Cardiovascular,High,null,abfss://data@datastoragea.dfs.core.windows.net/staging/diagnosis_raw/diagnosis_raw.csv,2026-05-04T13:33:37.059Z,2026-05-04
I63,Ischemic Stroke,Cardiovascular,High,null,abfss://data@datastoragea.dfs.core.windows.net/staging/diagnosis_raw/diagnosis_raw.csv,2026-05-04T13:33:37.059Z,2026-05-04
I61,Intracerebral Hemorrhage,Cardiovascular,High,null,abfss://data@datastoragea.dfs.core.windows.net/staging/diagnosis_raw/diagnosis_raw.csv,2026-05-04T13:33:37.059Z,2026-05-04
I48,Atrial Fibrillation,Cardiovascular,Medium,null,abfss://data@datastoragea.dfs.core.windows.net/staging/diagnosis_raw/diagnosis_raw.csv,2026-05-04T13:33:37.059Z,2026-05-04
E11,Type 2 Diabetes Mellitus,Metabolic,Medium,null,abfss://data@datastoragea.dfs.core.windows.net/staging/diagnosis_raw/diagnosis_raw.csv,2026-05-04T13:33:37.059Z,2026-05-04
E86,Dehydration,Metabolic,Low,null,abfss://data@datastoragea.dfs.core.windows.net/staging/diagnosis_raw/diagnosis_raw.csv,2026-05-04T13:33:37.059Z,2026-05-04
J18,Pneumonia,Respiratory,Medium,null,abfss://data@datastoragea.dfs.core.windows.net/staging/diagnosis_raw/diagnosis_raw.csv,2026-05-04T13:33:37.059Z,2026-05-04
J44,COPD,Respiratory,High,null,abfss://data@datastoragea.dfs.core.windows.net/staging/diagnosis_raw/diagnosis_raw.csv,2026-05-04T13:33:37.059Z,2026-05-04


In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
bronze_table = 'catalog.bronze.diagnosis_raw'
silver_table = 'catalog.silver.dim_diagnosis'
checkpoint_path = "abfss://data@datastoragea.dfs.core.windows.net/silver/dim_diagnosis/checkpoint/"

# Read from Bronze as a stream
df_diagnosis_bronze = spark.readStream.table(bronze_table)

df_diagnosis_clean = (
    df_diagnosis_bronze
        .drop(
            "_rescued_data",
            "_source_file",
            "_ingest_timestamp",
            "ingestion_date"
        )
        .withColumn("load_timestamp", current_timestamp())
)

from delta.tables import DeltaTable

# When diagnosis codes match, data gets merged. If not, data gets updated
def merge_dim_diagnosis(batch_df, batch_id):
    batch_df = batch_df.dropDuplicates(["diagnosis_code"])

    if not spark.catalog.tableExists(silver_table):
        # First run — create the table
        (batch_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(silver_table))   
        return
    
    # Load Delta table by name and upsert into Silver
    dim_diagnosis = DeltaTable.forName(spark, silver_table)

    (dim_diagnosis.alias("t")
        .merge(
            batch_df.alias("s"),
            "t.diagnosis_code = s.diagnosis_code"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    

In [0]:
(
    df_diagnosis_clean.writeStream
        .foreachBatch(merge_dim_diagnosis)  # for every batch, merge_dim_diagnosis is run
        .outputMode("update")
        .trigger(availableNow=True)
        .option("checkpointLocation", checkpoint_path)
        .start()
)


In [0]:
%sql
SELECT * FROM catalog.silver.dim_diagnosis;

diagnosis_code,diagnosis_desc,diagnosis_category,readmission_risk_level,load_timestamp
K92,Gastrointestinal Hemorrhage,Gastrointestinal,Medium,2026-05-21T14:40:33.366Z
I21,Acute Myocardial Infarction,Cardiovascular,High,2026-05-21T14:40:33.366Z
K70,Alcoholic Liver Disease,Gastrointestinal,High,2026-05-21T14:40:33.366Z
I61,Intracerebral Hemorrhage,Cardiovascular,High,2026-05-21T14:40:33.366Z
I10,Essential Hypertension,Cardiovascular,Medium,2026-05-21T14:40:33.366Z
Z51,Chemotherapy / Radiotherapy,Procedure,High,2026-05-21T14:40:33.366Z
A09,Infectious Gastroenteritis,Infectious,Low,2026-05-21T14:40:33.366Z
I48,Atrial Fibrillation,Cardiovascular,Medium,2026-05-21T14:40:33.366Z
A15,Pulmonary Tuberculosis,Respiratory,Medium,2026-05-21T14:40:33.366Z
E11,Type 2 Diabetes Mellitus,Metabolic,Medium,2026-05-21T14:40:33.366Z


In [0]:
# Data quality check
from pyspark.sql.functions import col, count

print("DIM DIAGNOSIS")
df = spark.read.table("catalog.silver.dim_diagnosis")
total = df.count()
print(f"Total rows: {total} (expected: 28)")

# Uniqueness
dup_id = total - df.select("diagnosis_code").distinct().count()
print(f"Duplicate diagnosis_codes: {'OK' if dup_id == 0 else dup_id}")

# Nulls
for c in ["diagnosis_code", "diagnosis_desc", "diagnosis_category", "readmission_risk_level"]:
    n = df.filter(col(c).isNull()).count()
    print(f"  {'OK' if n == 0 else 'CHECK'} {c}: {n} nulls")

# Valid risk levels
print("Risk level distribution:")
df.groupBy("readmission_risk_level").count().show()

DIM DIAGNOSIS
Total rows: 28 (expected: 28)
Duplicate diagnosis_codes: OK
  OK diagnosis_code: 0 nulls
  OK diagnosis_desc: 0 nulls
  OK diagnosis_category: 0 nulls
  OK readmission_risk_level: 0 nulls
Risk level distribution:
+----------------------+-----+
|readmission_risk_level|count|
+----------------------+-----+
|                  High|   15|
|                Medium|   11|
|                   Low|    2|
+----------------------+-----+

